In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/dejintime/moviereviews/testData.tsv/testData.tsv
/kaggle/input/datasets/dejintime/moviereviews/labeledTrainData.tsv/labeledTrainData.tsv
/kaggle/input/datasets/dejintime/moviereviews/unlabeledTrainData.tsv/unlabeledTrainData.tsv


# Part2

In [3]:
train_data = pd.read_csv("/kaggle/input/datasets/dejintime/moviereviews/labeledTrainData.tsv/labeledTrainData.tsv",
                        header = 0,
                        delimiter = "\t",
                        quoting = 3)
unlabeledTrain_data = pd.read_csv("/kaggle/input/datasets/dejintime/moviereviews/unlabeledTrainData.tsv/unlabeledTrainData.tsv",
                                 header = 0,
                                 delimiter = "\t",
                                 quoting = 3)
test_data = pd.read_csv("/kaggle/input/datasets/dejintime/moviereviews/testData.tsv/testData.tsv",
                       header = 0,
                       delimiter = "\t",
                       quoting = 3)
print(f'the nums of train reviews:{train_data["review"].size}\nthe nums of unlabeled_train reviews:{unlabeledTrain_data["review"].size}\nthe nums of test_data reviews:{test_data["review"].size}\n')

the nums of train reviews:25000
the nums of unlabeled_train reviews:50000
the nums of test_data reviews:25000



In [4]:
from bs4 import BeautifulSoup
import re
from nltk.corpus import stopwords
def CleanData_to_wordlist(raw_review, remove_stopwords=False):
    review_text = BeautifulSoup(raw_review).get_text()
    review_text = re.sub("[^a-zA-Z]", " ", review_text)
    words = review_text.lower().split()
    if remove_stopwords:
        words = [w for w in words if w not in stopwords.words("english")]
    return words

In [5]:
import nltk.data
nltk.download("punkt")
tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')
def review_to_sentences(review, tokenizer, remove_stopwords = False):
    raw_sentences = tokenizer.tokenize(review.strip())
    sentences = []
    for raw_sentence in raw_sentences:
        if len(raw_sentence) > 0:
            sentences.append(CleanData_to_wordlist(raw_sentence, remove_stopwords))
    return sentences

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [6]:
sentences = []
print(f"parsing sentences from training set\n")
for review in train_data["review"]:
    sentences += review_to_sentences(review, tokenizer)
print(f"parsing sentences from unlabeled_training set\n")
for review in unlabeledTrain_data["review"]:
    sentences += review_to_sentences(review, tokenizer)

parsing sentences from training set



/tmp/ipykernel_57/394599830.py:5: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  review_text = BeautifulSoup(raw_review).get_text()


parsing sentences from unlabeled_training set



In [7]:
len(sentences)

796172

In [8]:
print(sentences[0])

['with', 'all', 'this', 'stuff', 'going', 'down', 'at', 'the', 'moment', 'with', 'mj', 'i', 've', 'started', 'listening', 'to', 'his', 'music', 'watching', 'the', 'odd', 'documentary', 'here', 'and', 'there', 'watched', 'the', 'wiz', 'and', 'watched', 'moonwalker', 'again']


In [8]:
import logging
logging.basicConfig(format='%(asctime)s:%(levelname)s:%(message)s',level = logging.INFO)
num_features = 300
min_word_count = 40
num_workers = 4
context = 10
downsampling = 1e-3
from gensim.models import word2vec
print("training model...\n")
model = word2vec.Word2Vec(sentences, workers = num_workers, vector_size = num_features,min_count = min_word_count, window = context, sample = downsampling)
model.init_sims(replace=True)
model_name = "300features_40minwords_10context"
model.save(model_name)

2026-04-29 03:08:53,059:INFO:collecting all words and their counts
2026-04-29 03:08:53,060:INFO:PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2026-04-29 03:08:53,107:INFO:PROGRESS: at sentence #10000, processed 225664 words, keeping 17775 word types
2026-04-29 03:08:53,158:INFO:PROGRESS: at sentence #20000, processed 451738 words, keeping 24945 word types
2026-04-29 03:08:53,208:INFO:PROGRESS: at sentence #30000, processed 670859 words, keeping 30027 word types


training model...



2026-04-29 03:08:53,262:INFO:PROGRESS: at sentence #40000, processed 896841 words, keeping 34335 word types
2026-04-29 03:08:53,318:INFO:PROGRESS: at sentence #50000, processed 1116082 words, keeping 37751 word types
2026-04-29 03:08:53,374:INFO:PROGRESS: at sentence #60000, processed 1337544 words, keeping 40711 word types
2026-04-29 03:08:53,427:INFO:PROGRESS: at sentence #70000, processed 1560307 words, keeping 43311 word types
2026-04-29 03:08:53,480:INFO:PROGRESS: at sentence #80000, processed 1779516 words, keeping 45707 word types
2026-04-29 03:08:53,534:INFO:PROGRESS: at sentence #90000, processed 2003714 words, keeping 48121 word types
2026-04-29 03:08:53,586:INFO:PROGRESS: at sentence #100000, processed 2225465 words, keeping 50190 word types
2026-04-29 03:08:53,638:INFO:PROGRESS: at sentence #110000, processed 2444323 words, keeping 52058 word types
2026-04-29 03:08:53,692:INFO:PROGRESS: at sentence #120000, processed 2666488 words, keeping 54098 word types
2026-04-29 03:08:

In [9]:
model.wv.most_similar("man")

[('woman', 0.6216070652008057),
 ('lady', 0.5956532955169678),
 ('lad', 0.5488750338554382),
 ('monk', 0.5465677976608276),
 ('farmer', 0.5338476896286011),
 ('men', 0.5155428051948547),
 ('guy', 0.5140016078948975),
 ('person', 0.5131475925445557),
 ('businessman', 0.5046542882919312),
 ('priest', 0.5004774928092957)]

In [10]:
model.wv.most_similar("awful")

[('terrible', 0.7604466676712036),
 ('horrible', 0.7294919490814209),
 ('atrocious', 0.7265836000442505),
 ('dreadful', 0.7029842734336853),
 ('abysmal', 0.6990430355072021),
 ('horrendous', 0.6818488240242004),
 ('horrid', 0.6695940494537354),
 ('appalling', 0.6574757099151611),
 ('lousy', 0.5976453423500061),
 ('amateurish', 0.5896868705749512)]

# Part 3

#### Vector Averaging

In [12]:
def makeFeatureVec(words, model, num_features):
    feature_vecs = np.zeros((num_features, ), dtype="float32")
    nwords = 0
    index2word_set = set(model.wv.index_to_key)
    for word in words:
        if word in index2word_set:
            nwords = nwords + 1
            feature_vecs = np.add(feature_vecs, model.wv[word])
    feature_vecs = np.divide(feature_vecs, nwords)
    return feature_vecs

In [13]:
def getAvgFeatureVecs(reviews, model, num_features):
    review_feature_vecs = np.zeros((len(reviews), num_features), dtype="float32")
    counter = 0
    for review in reviews:
        if counter%1000. == 0.:
            print(f"review:{counter} / {len(reviews)}")
        review_feature_vecs[counter] = makeFeatureVec(review, model, num_features)
        counter = counter + 1
    return review_feature_vecs

### Calculate average feature vectors

In [17]:
print("Creating average feature vecs for train reviews")
clean_train_reviews = []
for review in train_data["review"]:
    clean_train_reviews.append(CleanData_to_wordlist(review, remove_stopwords = True))

Creating average feature vecs for train reviews


In [15]:
trainDataVecs = getAvgFeatureVecs(clean_train_reviews, model, num_features)

review:0 / 25000
review:1000 / 25000
review:2000 / 25000
review:3000 / 25000
review:4000 / 25000
review:5000 / 25000
review:6000 / 25000
review:7000 / 25000
review:8000 / 25000
review:9000 / 25000
review:10000 / 25000
review:11000 / 25000
review:12000 / 25000
review:13000 / 25000
review:14000 / 25000
review:15000 / 25000
review:16000 / 25000
review:17000 / 25000
review:18000 / 25000
review:19000 / 25000
review:20000 / 25000
review:21000 / 25000
review:22000 / 25000
review:23000 / 25000
review:24000 / 25000


In [19]:
print("Creating average feature vecs for test reviews")
clean_test_reviews = []
for review in test_data["review"]:
    clean_test_reviews.append(CleanData_to_wordlist(review, remove_stopwords = True))

Creating average feature vecs for test reviews


In [17]:
testDataVecs = getAvgFeatureVecs(clean_test_reviews, model, num_features)

review:0 / 25000
review:1000 / 25000
review:2000 / 25000
review:3000 / 25000
review:4000 / 25000
review:5000 / 25000
review:6000 / 25000
review:7000 / 25000
review:8000 / 25000
review:9000 / 25000
review:10000 / 25000
review:11000 / 25000
review:12000 / 25000
review:13000 / 25000
review:14000 / 25000
review:15000 / 25000
review:16000 / 25000
review:17000 / 25000
review:18000 / 25000
review:19000 / 25000
review:20000 / 25000
review:21000 / 25000
review:22000 / 25000
review:23000 / 25000
review:24000 / 25000


In [18]:
from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier(n_estimators = 100)
print("Fitting a random forest to labeled training data...")
forest = forest.fit(trainDataVecs, train_data["sentiment"])
result = forest.predict(testDataVecs)
output = pd.DataFrame( data={"id":test_data["id"], "sentiment":result} )
os.makedirs("/kaggle/working/output", exist_ok = True)
output.to_csv( "/kaggle/working/output/Word2Vec_AverageVectors.csv", index=False, quoting=3 )

Fitting a random forest to labeled training data...


#### Clustering

In [11]:
from sklearn.cluster import KMeans
import time
start = time.time()
word_vectors = model.wv.vectors
num_clusters = word_vectors.shape[0] // 5
kmeans_clustering = KMeans(n_clusters = num_clusters)
idx = kmeans_clustering.fit_predict(word_vectors)
end = time.time()
elapsed = end - start
print("Time taken for K Means clustering: ", elapsed, "seconds.")

Time taken for K Means clustering:  59.72939872741699 seconds.


In [12]:
word_centroid_map = dict(zip( model.wv.index_to_key, idx ))

In [13]:
for cluster in range(10):
    print(f"\nCluster {cluster}")
    words = [word for word, c in word_centroid_map.items() if c == cluster]
    print(words)


Cluster 0
['scariest', 'dumbest', 'coolest', 'stupidest', 'strangest', 'lamest', 'poorest', 'cheesiest', 'weirdest', 'ugliest', 'creepiest', 'silliest']

Cluster 1
['george', 'lo', 'naish', 'cobb', 'carrol']

Cluster 2
['everywhere', 'destroying', 'attacking', 'mole', 'pod', 'starving', 'observing']

Cluster 3
['bliss', 'ceremonies']

Cluster 4
['offensive', 'insulting', 'disrespectful', 'objectionable', 'pandering']

Cluster 5
['carol', 'graham', 'hart', 'fisher', 'fleming', 'raul', 'isaac', 'burnett']

Cluster 6
['bubble', 'gum', 'clap']

Cluster 7
['scotland', 'dc', 'alabama', 'arctic', 'alaska']

Cluster 8
['theft', 'biological', 'bid', 'ownership', 'risking']

Cluster 9
['understated', 'nuanced', 'shakespearean', 'literate', 'duris', 'histrionic']


In [14]:
def create_bag_of_centroids( wordlist, word_centroid_map ):
    num_centroids = max( word_centroid_map.values() ) + 1
    bag_of_centroids = np.zeros( num_centroids, dtype="float32" )
    for word in wordlist:
        if word in word_centroid_map:
            index = word_centroid_map[word]
            bag_of_centroids[index] += 1
    return bag_of_centroids


In [20]:
train_centroids = np.zeros( (train_data["review"].size, num_clusters), dtype="float32" )
counter = 0
for review in clean_train_reviews:
    train_centroids[counter] = create_bag_of_centroids( review, word_centroid_map )
    counter += 1

test_centroids = np.zeros(( test_data["review"].size, num_clusters), dtype="float32" )
counter = 0
for review in clean_test_reviews:
    test_centroids[counter] = create_bag_of_centroids( review, word_centroid_map )
    counter += 1

In [25]:
from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier(n_estimators = 100)
print("Fitting a random forest to labeled training data...")
forest = forest.fit(train_centroids,train_data["sentiment"])
result = forest.predict(test_centroids)

output = pd.DataFrame(data={"id":test_data["id"], "sentiment":result})
os.makedirs("/kaggle/working/output", exist_ok = True)
output.to_csv( "/kaggle/working/output/BagOfCentroids.csv", index=False, quoting=3 )

Fitting a random forest to labeled training data...
